# Raw Data Exploration & Quality Audit: Pakistan Elections (1970–2024)
**Author:** Qudrat Ullah | **Trainee ID:** TRN-2026-000363

**Objective:** Inspect the raw operational dataset (`pakistan-elections-database-1970-2024.csv`), document its schema, and measure structural missingness, duplicates, invalid entries, and category spelling variants before building the reproducible cleaning pipeline[cite: 2, 3].

## 1. Environment Setup & File Paths
Import `pandas`, `numpy`, and `pathlib` to configure OS-independent paths from `notebooks/` to `data/raw/` and `output/reports/`, ensuring raw data is never modified[cite: 3, 4].

In [11]:

from pathlib import Path
import numpy as np
import pandas as pd

RAW_PATH = Path("../data/raw/pakistan-elections-database-1970-2024.csv")
REPORT_DIR = Path("../output/reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Raw Dataset (Read-Only)
Load `pakistan-elections-database-1970-2024.csv` using `encoding="utf-8-sig"` to strip the hidden UTF-8 BOM character (`\ufeff`) from the first column (`Year`) and inspect dataset dimensions and sample rows.

In [6]:
df = pd.read_csv(RAW_PATH, encoding="utf-8-sig", low_memory=False)
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head(5)

Rows: 24,585 | Columns: 17


,Year,Constituency,NA,Province,District,Division,Party,Candidate Name,Regions 11,Regions 12,Regions 5,Registered Voters,Rejected Votes,Six Parties,Turnout N,Zones 20,Votes
0,1970,NA-1 - Peshawar,NA-1,KPK,Peshawar,Peshawar,JI,Hussain Ahmad Qazi,Peshawar Valley,Peshawar Valley,KPK,NaN,NaN,Religious Parties,NaN,Zone-3 (Peshawar Valley) Peshawar & Mardan Div...,2475.0
1,1970,NA-1 - Peshawar,NA-1,KPK,Peshawar,Peshawar,JUI-W.P,Molana Abdul Qayoom Popalzai,Peshawar Valley,Peshawar Valley,KPK,NaN,NaN,Religious Parties,NaN,Zone-3 (Peshawar Valley) Peshawar & Mardan Div...,13248.0
2,1970,NA-1 - Peshawar,NA-1,KPK,Peshawar,Peshawar,NAP-W,Arbab Sikandar Khan,Peshawar Valley,Peshawar Valley,KPK,NaN,NaN,Province Parties,NaN,Zone-3 (Peshawar Valley) Peshawar & Mardan Div...,21622.0
3,1970,NA-1 - Peshawar,NA-1,KPK,Peshawar,Peshawar,PML,Alfred Peter,Peshawar Valley,Peshawar Valley,KPK,NaN,NaN,Other Parties,NaN,Zone-3 (Peshawar Valley) Peshawar & Mardan Div...,2499.0
4,1970,NA-1 - Peshawar,NA-1,KPK,Peshawar,Peshawar,PML-Q,Khan Abdul Qayoom Khan,Peshawar Valley,Peshawar Valley,KPK,148708.0,1388.0,PML (s)/IJI/PNA,62.15402,Zone-3 (Peshawar Valley) Peshawar & Mardan Div...,27215.0


## 3. Schema Profiling & Missing Value Audit
Inspect column data types, non-null counts, missing percentages, unique value counts, and sample values. Export the audit table to `output/reports/raw_schema_audit.csv` as the baseline for `docs/schema.md`[cite: 3, 8].

In [7]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
    "sample_value": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
audit.to_csv(REPORT_DIR / "raw_schema_audit.csv")
audit

,dtype,missing_count,missing_pct,unique_values,sample_value
Year,int64,0,0.00,11,1970
Constituency,str,0,0.00,1216,NA-1 - Peshawar
NA,str,0,0.00,272,NA-1
Province,str,34,0.14,5,KPK
District,str,0,0.00,148,Peshawar
Division,str,0,0.00,32,Peshawar
Party,str,2,0.01,475,JI
Candidate Name,str,3354,13.64,17715,Hussain Ahmad Qazi
Regions 11,str,1,0.00,11,Peshawar Valley
Regions 12,str,0,0.00,12,Peshawar Valley


## 4. Duplicate Detection & Multi-Constituency Domain Check
Check for exact full-row copies versus composite-key repeats (`Year + Constituency + Party + Candidate Name + Votes`), while preserving candidates who legitimately contest multiple constituencies in the same election year.

In [8]:
print("Exact full-row duplicates:", df.duplicated().sum())

key_cols = ["Year", "Constituency", "Candidate Name"]
print("Key duplicates (Year + Constituency + Candidate):", df.duplicated(subset=key_cols).sum())

# Inspect why key duplicates happen (e.g., missing Candidate Name in 2018)
print("\nMissing Candidate Name by Year:")
print(df.groupby("Year")["Candidate Name"].apply(lambda s: s.isna().sum()))

Exact full-row duplicates: 0
Key duplicates (Year + Constituency + Candidate): 3170

Missing Candidate Name by Year:
Year
1970       1
1977       0
1988       0
1990       0
1993       0
1997       0
2002       0
2008       0
2013       0
2018    3353
2024       0
Name: Candidate Name, dtype: int64


In [9]:
print("Exact full-row duplicates:", df.duplicated().sum())

# 1. Legitimate multi-constituency candidates in the same year
non_null_cand = df[df["Candidate Name"].notna()]
multi_seat = non_null_cand.groupby(["Year", "Candidate Name"])["Constituency"].nunique()
print("Candidates running in multiple constituencies (same year):", int((multi_seat > 1).sum()))

# 2. True repeated entries in the SAME Year + Constituency + Party + Candidate + Votes
key_cols = ["Year", "Constituency", "Party", "Candidate Name", "Votes"]
same_seat_dups = non_null_cand[non_null_cand.duplicated(subset=key_cols, keep=False)]
print("Repeated entries in same seat (partial-null row copies):", len(same_seat_dups))
same_seat_dups[["Year", "Constituency", "Party", "Candidate Name", "Registered Voters", "Votes"]].head(6)

Exact full-row duplicates: 0
Candidates running in multiple constituencies (same year): 964
Repeated entries in same seat (partial-null row copies): 123


,Year,Constituency,Party,Candidate Name,Registered Voters,Votes
19638,2024,NA-107 - Toba Tek Singh 3,IND,Muhammad Saleem,187503.0,379.0
19639,2024,NA-107 - Toba Tek Singh 3,IND,Muhammad Saleem,338780.0,379.0
19641,2024,NA-107 - Toba Tek Singh 3,IND,Muhammad Saleem,708151.0,379.0
19728,2024,NA-111 - Nankana Sahib 1,IND,Muhammad Saleem,187503.0,2148.0
19730,2024,NA-111 - Nankana Sahib 1,IND,Muhammad Saleem,338780.0,2148.0
19731,2024,NA-111 - Nankana Sahib 1,IND,Muhammad Saleem,708151.0,2148.0


## 5. Numeric Distributions & Domain Validity Checks
Examine summary statistics (`min`, `max`, quartiles) and quantify impossible values: negative vote counts, `Registered Voters <= 0`, `Turnout N` outside `[0, 100]`, and `Votes > Registered Voters`[cite: 3].

In [10]:
num_cols = ["Registered Voters", "Rejected Votes", "Turnout N", "Votes"]
display(df[num_cols].describe().T)

validity_issues = {
    "Negative Votes": int((df["Votes"] < 0).sum()),
    "Negative Rejected Votes": int((df["Rejected Votes"] < 0).sum()),
    "Registered Voters <= 0": int((df["Registered Voters"] <= 0).sum()),
    "Turnout N < 0 or > 100": int(((df["Turnout N"] < 0) | (df["Turnout N"] > 100)).sum()),
    "Votes > Registered Voters": int((df["Votes"] > df["Registered Voters"]).sum()),
}
pd.Series(validity_issues, name="issue_count")

,count,mean,std,min,25%,50%,75%,max
Registered Voters,2805.0,295345.932977,144128.353996,0.0,216140.000000,269306.000000,352085.000000,3.022628e+06
Rejected Votes,2631.0,3655.803877,2939.006000,0.0,1459.500000,2784.000000,5181.000000,2.556200e+04
Turnout N,2627.0,45.785052,13.770434,0.0,38.678162,46.831488,54.315094,9.780666e+01
Votes,24392.0,13731.363193,26005.407906,0.0,200.000000,1035.000000,12810.250000,2.174270e+05


Negative Votes               0
Negative Rejected Votes      0
Registered Voters <= 0       6
Turnout N < 0 or > 100       0
Votes > Registered Voters    4
Name: issue_count, dtype: int64

## 6. Structural Missingness & Categorical Spelling Variants
Verify that constituency-level fields (`Registered Voters`, `Rejected Votes`, `Turnout N`) are recorded once per constituency per election year, and inspect top `Province`, `Party`, and `Six Parties` labels for spelling inconsistencies[cite: 3].

In [12]:
print("Non-null Registered Voters per Year (recorded once per constituency):")
print(df.groupby("Year")["Registered Voters"].apply(lambda s: s.notna().sum()))

for col in ["Province", "Party", "Six Parties"]:
    print(f"\n=== {col} (Top 15 of {df[col].nunique()} unique) ===")
    print(df[col].value_counts(dropna=False).head(15))

Non-null Registered Voters per Year (recorded once per constituency):
Year
1970    138
1977    200
1988    206
1990    206
1993    206
1997    206
2002    272
2008    270
2013    271
2018    270
2024    560
Name: Registered Voters, dtype: int64

=== Province (Top 15 of 5 unique) ===
Province
Punjab         12073
Sindh           6175
KPK             4528
Balochistan     1662
ICT              113
NaN               34
Name: count, dtype: int64

=== Party (Top 15 of 475 unique) ===
Party
IND                                  11788
PPP                                   2016
PML-N                                 1371
MQM                                   1149
PTI                                    703
TLP                                    395
JUI-F                                  356
ANP                                    328
IJI                                    324
Muttahidda Majlis-e-Amal Pakistan      284
Pakistan Muslim League                 252
IND (PTI)                             

## 7. Exploratory Summary Statistics & Vectorized Transformations
Prototype Pandas `groupby` aggregations (voter turnout and candidate counts by election year) and vectorized NumPy operations (`np.log1p` on votes and `np.where` turnout tiers) prior to moving them into `src/pipeline_part_a.py`[cite: 3].

In [13]:
# 1. Pandas groupby: average turnout and total votes by election year
yearly_summary = df.groupby("Year", as_index=False).agg(
    total_candidates=("Votes", "count"),
    total_votes=("Votes", "sum"),
    avg_turnout_pct=("Turnout N", "mean"),
)
display(yearly_summary)

# 2. NumPy vectorized operations: log-transformed votes & turnout tier classification
votes_filled = df["Votes"].fillna(0).to_numpy()
log_votes = np.log1p(votes_filled)
turnout_tier = np.where(
    df["Turnout N"].isna(), "Unknown",
    np.where(df["Turnout N"] >= 50.0, "High (>=50%)", "Moderate/Low (<50%)")
)
print("Sample log1p(Votes):", np.round(log_votes[:5], 3))
print("Turnout tiers:\n", pd.Series(turnout_tier).value_counts())

,Year,total_candidates,total_votes,avg_turnout_pct
0,1970,797,15810620.0,64.058247
1,1977,566,16894991.0,46.887712
2,1988,1292,19692186.0,44.262311
3,1990,1309,21163831.0,46.149661
4,1993,1431,20453298.0,41.644721
5,1997,1761,19240574.0,36.042647
6,2002,2066,29236687.0,40.918246
7,2008,2204,34615929.0,44.762826
8,2013,4454,45116452.0,52.629488
9,2018,3353,52741186.0,51.029694


Sample log1p(Votes): [ 7.814  9.492  9.982  7.824 10.212]
Turnout tiers:
 Unknown                21958
Moderate/Low (<50%)     1546
High (>=50%)            1081
Name: count, dtype: int64
